# NIDS — Model Training (CICIDS2017)

This notebook trains the classifier that the NIDS dashboard loads, and exports
the five files the app expects. Run the cells top to bottom.

**What it does**
1. Loads the CICIDS2017 CSVs from Google Drive
2. Cleans them and maps their columns onto the 20 features the app uses
3. Collapses the many CICIDS attack labels into the 9 classes the app knows
4. Handles the heavy class imbalance (about 80% of traffic is benign)
5. Trains **XGBoost** and compares it against Random Forest and Decision Tree
6. Evaluates the best model
7. Exports `model.joblib`, `scaler.joblib`, `feature_columns.json`,
   `label_encoder.joblib`, `metrics.json`

**Before you start**
- Runtime → Change runtime type → **High-RAM** if available. The dataset is large.
- You do **not** need a GPU. XGBoost on this data trains fine on CPU.

**A note on the numbers.** Accuracy on this dataset is high, but that reflects
CICIDS2017 held-out data, not a guarantee on live, drifting traffic. Report the
per-class recall honestly — especially for rare classes like Infiltration —
rather than a single headline number.

**Why XGBoost and 20 features**
XGBoost models are small (they deploy under GitHub's 100 MB limit) and handle
tabular data well. The 20 features are the ones the live-capture path can also
produce, so the same model works on uploaded CSVs and on live traffic without
retraining.

## 1. Install and import

In [ ]:
# Colab has most of these. xgboost and imbalanced-learn may need installing.
!pip install -q xgboost imbalanced-learn scikit-learn joblib shap

import os, json, glob, warnings, time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, roc_curve, auc, classification_report)
from sklearn.preprocessing import label_binarize
from xgboost import XGBClassifier
import joblib
import shap

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
print("Imports OK")

## 2. Get the dataset

This mounts your Google Drive and downloads CICIDS2017 into
`MyDrive/cicids2017/` the first time only; later runs find the files already
there and skip the download.

The data comes from **Kaggle** (`chethuhn/network-intrusion-dataset`), which is
CC0-licensed, open, and gives the eight CSV files this notebook expects. Kaggle
is used because it is reliable; the CIC website hosts the same data but its
direct link changes and often returns an error instead of the zip, so it is only
a fallback.

**One-time Kaggle setup:** on kaggle.com, go to your avatar -> Settings -> API ->
**Create New Token**. That downloads a `kaggle.json` file. The cell below asks
you to upload it, then handles the rest.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import zipfile, urllib.request, shutil

# Where the CSVs live in your Drive. Change this if you want a different folder.
DATA_DIR = "/content/drive/MyDrive/cicids2017"
os.makedirs(DATA_DIR, exist_ok=True)

# Model + SHAP outputs go here, on Drive, so nothing large touches local disk
# and everything survives the session ending.
OUT_DIR = "/content/drive/MyDrive/nids_model_out"
os.makedirs(OUT_DIR, exist_ok=True)

def have_csvs():
    return len(glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True)) >= 8

# ---------------------------------------------------------------------------
# Getting CICIDS2017. The most reliable source is Kaggle: the mirror
# "chethuhn/network-intrusion-dataset" is CC0-licensed, open, and gives the
# eight CSV files this notebook expects. The CIC website also hosts the data,
# but its direct link changes and sometimes returns an error page instead of
# the zip, so Kaggle is tried first and CIC is only a fallback.
#
# For Kaggle you need a free API token (one-time):
#   kaggle.com -> your avatar -> Settings -> API -> Create New Token
# That downloads kaggle.json. Upload it when the cell below asks.
# ---------------------------------------------------------------------------

if have_csvs():
    print("CSVs already in Drive, skipping download.")
else:
    got = False

    # --- Option 1: Kaggle (recommended, this is the reliable path) ---
    try:
        print("Downloading CICIDS2017 from Kaggle into your Drive...")
        !pip install -q kaggle

        # Upload kaggle.json if it is not already here.
        if not os.path.exists("/root/.kaggle/kaggle.json"):
            print("\nUpload your kaggle.json (from Kaggle -> Settings -> API):")
            from google.colab import files
            files.upload()
            os.makedirs("/root/.kaggle", exist_ok=True)
            shutil.copy("kaggle.json", "/root/.kaggle/kaggle.json")
            os.chmod("/root/.kaggle/kaggle.json", 0o600)

        !kaggle datasets download -d chethuhn/network-intrusion-dataset -p "{DATA_DIR}" --unzip
        got = have_csvs()
        if got:
            print("Downloaded from Kaggle.")
    except Exception as e:
        print("Kaggle download did not complete:", e)

    # --- Option 2: CIC direct link (fallback, may be down) ---
    if not got:
        try:
            print("\nFalling back to the CIC direct link...")
            CIC_URL = ("http://cicresearch.ca/CICDataset/CIC-IDS-2017/Dataset/"
                       "CIC-IDS-2017/CSVs/MachineLearningCSV.zip")
            zip_path = os.path.join(DATA_DIR, "_download.zip")
            req = urllib.request.urlopen(CIC_URL, timeout=60)
            total = int(req.headers.get("Content-Length", 0)); done = 0
            with open(zip_path, "wb") as out:
                while True:
                    chunk = req.read(1024 * 512)
                    if not chunk:
                        break
                    out.write(chunk); done += len(chunk)
                    pct = min(100, done * 100 // total) if total else 0
                    print(f"\r  {pct}%  ({done // (1024*1024)} MB)", end="")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(DATA_DIR)
            os.remove(zip_path)
            got = have_csvs()
        except Exception as e:
            if os.path.exists(os.path.join(DATA_DIR, "_download.zip")):
                os.remove(os.path.join(DATA_DIR, "_download.zip"))
            print("\nCIC fallback also failed:", e)

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
print(f"\nFound {len(csv_files)} CSV files:")
for f in csv_files:
    print("  ", os.path.basename(f))

assert csv_files, (
    "No CSVs found. Get a Kaggle API token (kaggle.com -> Settings -> API -> "
    "Create New Token), re-run this cell, and upload kaggle.json when asked. "
    "The Kaggle dataset is 'chethuhn/network-intrusion-dataset'."
)

### If the download fails

The most common cause is a missing or expired Kaggle token. Fix it and re-run
the cell above:

1. On kaggle.com: avatar -> Settings -> API -> **Create New Token**, which
   downloads `kaggle.json`.
2. Re-run the cell above and upload that file when prompted.

If you would rather download by hand, get the CSVs from the Kaggle dataset
["chethuhn/network-intrusion-dataset"](https://www.kaggle.com/datasets/chethuhn/network-intrusion-dataset)
or from <https://www.unb.ca/cic/datasets/ids-2017.html>, then upload the eight
CSV files into `drive/MyDrive/cicids2017/` using the Colab file browser (the
folder icon on the left) and re-run the cell above.

## 3. The feature contract

These 20 names, in this order, are copied from the app's `core/schema.py`. The
app aligns every input to this exact list, so the model **must** be trained on
these columns in this order. The cell also defines how CICIDS2017's raw column
names (which vary in spacing and casing between dataset mirrors) map onto them.

In [ ]:
# The 20 features the app expects, in order. Do not reorder.
FEATURE_COLUMNS = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Fwd Packet Length Max",
    "Fwd Packet Length Mean",
    "Bwd Packet Length Max",
    "Bwd Packet Length Mean",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "Flow IAT Std",
    "Flow IAT Max",
    "Fwd IAT Mean",
    "Bwd IAT Mean",
    "Min Packet Length",
    "Max Packet Length",
    "Packet Length Mean",
    "SYN Flag Count",
]

def normalise(name):
    # Loose key so "Flow Duration", " Flow Duration", "flow_duration" all match.
    return " ".join(str(name).strip().lower().replace("_", " ").split())

# CICIDS2017 uses a few names that differ from ours. Map the loose key of each
# CICIDS column onto our schema name. Anything not listed here matches by its
# normalised name directly.
CICIDS_ALIASES = {
    "total length of fwd packet": "Total Length of Fwd Packets",
    "total length of bwd packet": "Total Length of Bwd Packets",
    "fwd packet length max": "Fwd Packet Length Max",
    "bwd packet length max": "Bwd Packet Length Max",
    "min packet length": "Min Packet Length",
    "max packet length": "Max Packet Length",
    "packet length min": "Min Packet Length",
    "packet length max": "Max Packet Length",
    "syn flag count": "SYN Flag Count",
    "syn flag cnt": "SYN Flag Count",
    "flow bytes/s": "Flow Bytes/s",
    "flow packets/s": "Flow Packets/s",
    "flow pkts/s": "Flow Packets/s",
    "flow byts/s": "Flow Bytes/s",
    "tot fwd pkts": "Total Fwd Packets",
    "tot bwd pkts": "Total Backward Packets",
    "totlen fwd pkts": "Total Length of Fwd Packets",
    "totlen bwd pkts": "Total Length of Bwd Packets",
    "fwd pkt len max": "Fwd Packet Length Max",
    "fwd pkt len mean": "Fwd Packet Length Mean",
    "bwd pkt len max": "Bwd Packet Length Max",
    "bwd pkt len mean": "Bwd Packet Length Mean",
    "pkt len min": "Min Packet Length",
    "pkt len max": "Max Packet Length",
    "pkt len mean": "Packet Length Mean",
    "flow iat mean": "Flow IAT Mean",
    "flow iat std": "Flow IAT Std",
    "flow iat max": "Flow IAT Max",
    "fwd iat mean": "Fwd IAT Mean",
    "bwd iat mean": "Bwd IAT Mean",
}

# Build a lookup from every schema feature's own normalised key to itself, so a
# direct-named column is found even if it is not in the alias table.
SCHEMA_BY_KEY = {normalise(c): c for c in FEATURE_COLUMNS}

def resolve_column(raw_name):
    # Return the schema feature this raw CICIDS column maps to, or None.
    key = normalise(raw_name)
    if key in CICIDS_ALIASES:
        return CICIDS_ALIASES[key]
    if key in SCHEMA_BY_KEY:
        return SCHEMA_BY_KEY[key]
    return None

print(f"{len(FEATURE_COLUMNS)} features pinned to the app schema")

## 4. Load and merge the CSVs

In [ ]:
frames = []
for path in csv_files:
    # CICIDS files have a stray whitespace in many headers; low_memory keeps
    # dtype guessing sane on the big files.
    df = pd.read_csv(path, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    frames.append(df)
    print(f"  {os.path.basename(path):45} {df.shape}")

raw = pd.concat(frames, ignore_index=True)
del frames
print("\nMerged:", raw.shape)

# The label column is usually the last one and is called 'Label' (sometimes with
# stray spacing, already stripped above).
label_col = "Label" if "Label" in raw.columns else raw.columns[-1]
print("Label column:", label_col)
print("\nRaw label counts:")
print(raw[label_col].value_counts())

## 5. Map columns onto the 20 features

Every CICIDS column is checked against the schema. Features that are found are
kept; any the dataset does not provide are filled with zero (and reported, so
you know). This is the same alignment the app does at inference time, so
training and serving see identical inputs.

In [ ]:
resolved = {}   # schema feature -> the raw column that supplies it
for col in raw.columns:
    if col == label_col:
        continue
    target = resolve_column(col)
    if target and target not in resolved:
        resolved[target] = col

found = [f for f in FEATURE_COLUMNS if f in resolved]
missing = [f for f in FEATURE_COLUMNS if f not in resolved]

print(f"Matched {len(found)}/{len(FEATURE_COLUMNS)} features")
if missing:
    print("Filled with zero (not in this dataset):")
    for m in missing:
        print("   ", m)

# Build the feature matrix in the exact schema order.
X = pd.DataFrame(index=raw.index)
for feat in FEATURE_COLUMNS:
    if feat in resolved:
        X[feat] = pd.to_numeric(raw[resolved[feat]], errors="coerce")
    else:
        X[feat] = 0.0

# CICIDS has infinities in the rate columns wherever duration is zero.
X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
print("\nFeature matrix:", X.shape)

## 6. Collapse labels into the 9 app classes

CICIDS2017 has many attack labels. The app knows nine. This maps the raw labels
onto those nine. Anything unrecognised is dropped rather than guessed at.

In [ ]:
# App classes: BENIGN, DoS, DDoS, PortScan, FTP-Patator, SSH-Patator,
#              Web Attack, Bot, Infiltration
def map_label(raw_label):
    s = str(raw_label).strip().lower()
    if s in ("benign", "normal"):
        return "BENIGN"
    if "ddos" in s:
        return "DDoS"
    if "dos" in s:                       # DoS Hulk, GoldenEye, slowloris, ...
        return "DoS"
    if "portscan" in s or "port scan" in s:
        return "PortScan"
    if "ftp" in s and "patator" in s:
        return "FTP-Patator"
    if "ssh" in s and "patator" in s:
        return "SSH-Patator"
    if "web attack" in s or "brute force" in s or "xss" in s or "sql injection" in s:
        return "Web Attack"
    if "bot" in s:
        return "Bot"
    if "infiltration" in s:
        return "Infiltration"
    return None                          # unknown -> dropped

y_raw = raw[label_col].apply(map_label)

# Drop rows whose label did not map, and align X to them.
keep = y_raw.notna()
X = X[keep].reset_index(drop=True)
y = y_raw[keep].reset_index(drop=True)
del raw

print("Mapped class counts:")
print(y.value_counts())
print(f"\nBenign share: {(y == 'BENIGN').mean() * 100:.1f}%")

## 7. The class imbalance, and how this notebook handles it

Roughly 80% of the data is benign. A model that predicts "benign" for everything
would score about 80% accuracy while catching **no attacks at all**, so accuracy
alone is misleading here — the per-class recall in the evaluation is what matters.

This notebook handles the imbalance two ways at once:

- **Downsample benign** so it does not swamp the attacks, while keeping enough of
  it to represent normal traffic.
- **Class weights** in the model, so the rare attack classes still count.

It deliberately does **not** use SMOTE by default. Synthesising fake flows for
classes like Infiltration (which has very few real samples) tends to invent
patterns that do not occur in real traffic, and a model that learns them looks
good on paper and fails in practice. Downsampling plus class weights is the more
honest choice. If you want to try SMOTE, there is a commented cell after this one.

In [ ]:
# Downsample benign to at most 3x the largest attack class, so normal traffic
# still dominates (as it should) without drowning everything else.
counts = y.value_counts()
largest_attack = counts.drop("BENIGN").max()
benign_cap = int(largest_attack * 3)

benign_idx = y[y == "BENIGN"].index
attack_idx = y[y != "BENIGN"].index

if len(benign_idx) > benign_cap:
    keep_benign = np.random.RandomState(42).choice(benign_idx, benign_cap, replace=False)
    keep_all = np.concatenate([keep_benign, attack_idx.values])
    keep_all.sort()
    X = X.loc[keep_all].reset_index(drop=True)
    y = y.loc[keep_all].reset_index(drop=True)

print("After downsampling benign:")
print(y.value_counts())
print(f"\nBenign share now: {(y == 'BENIGN').mean() * 100:.1f}%")

In [ ]:
# OPTIONAL: SMOTE instead of / on top of downsampling. Off by default.
# Uncomment to try it. Expect a better-looking confusion matrix and, often,
# worse behaviour on real traffic for the tiny classes.
#
# from imblearn.over_sampling import SMOTE
# sm = SMOTE(random_state=42, k_neighbors=3)
# X, y = sm.fit_resample(X, y)
# print(y.value_counts())

## 8. Encode labels, scale features, split

In [ ]:
# Label encoder: the app calls inverse_transform on predictions, so this exact
# object is exported and reused at serving time.
label_encoder = LabelEncoder()
y_enc = label_encoder.fit_transform(y)
print("Classes:", list(label_encoder.classes_))

# Scaler: the app calls transform on inputs before predict, so this exact object
# is exported too. Fit on the training split only, never on the test data.
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.25, random_state=42, stratify=y_enc
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train_scaled.shape, " Test:", X_test_scaled.shape)

## 9. Train and compare models

XGBoost is the one we deploy. Random Forest and Decision Tree are trained too, so
the Model Performance page can show an honest comparison rather than a single
number with nothing to weigh it against.

In [ ]:
# Per-sample weights so the rare classes count during training.
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight("balanced", y_train)

n_classes = len(label_encoder.classes_)
comparison = []
models = {}

# --- XGBoost (the one we deploy) ---
print("Training XGBoost...")
t0 = time.time()
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=n_classes,
    tree_method="hist",
    eval_metric="mlogloss",
    n_jobs=-1,
    random_state=42,
)
xgb.fit(X_train_scaled, y_train, sample_weight=sample_weights)
xgb_time = time.time() - t0
models["XGBoost"] = xgb
print(f"  done in {xgb_time:.0f}s")

# --- Random Forest ---
print("Training Random Forest...")
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=120, max_depth=20, class_weight="balanced",
    n_jobs=-1, random_state=42,
)
rf.fit(X_train_scaled, y_train)
rf_time = time.time() - t0
models["RandomForest"] = rf
print(f"  done in {rf_time:.0f}s")

# --- Decision Tree (a simple baseline) ---
print("Training Decision Tree...")
t0 = time.time()
dt = DecisionTreeClassifier(max_depth=20, class_weight="balanced", random_state=42)
dt.fit(X_train_scaled, y_train)
dt_time = time.time() - t0
models["DecisionTree"] = dt
print(f"  done in {dt_time:.0f}s")

# Score each one.
for name, model in models.items():
    pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, pred)
    _, _, f1, _ = precision_recall_fscore_support(
        y_test, pred, average="weighted", zero_division=0)
    tt = {"XGBoost": xgb_time, "RandomForest": rf_time, "DecisionTree": dt_time}[name]
    comparison.append({"model": name, "accuracy": round(float(acc), 4),
                       "f1": round(float(f1), 4), "train_seconds": round(tt, 1)})

comparison_df = pd.DataFrame(comparison).sort_values("f1", ascending=False)
print("\nComparison:")
print(comparison_df.to_string(index=False))

## 10. Pick the best model and evaluate it

In [ ]:
best_name = comparison_df.iloc[0]["model"]
best_model = models[best_name]
print(f"Best model: {best_name}\n")

y_pred = best_model.predict(X_test_scaled)
y_proba = best_model.predict_proba(X_test_scaled)
labels = list(label_encoder.classes_)

# Headline numbers.
acc = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="weighted", zero_division=0)
print(f"Accuracy : {acc*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall   : {recall*100:.2f}%")
print(f"F1       : {f1*100:.2f}%\n")

# Per-class report. Read the recall column for the rare classes — that is where
# a model that looks good overall can still be quietly failing.
print(classification_report(y_test, y_pred, target_names=labels, zero_division=0))

## 11. Build `metrics.json`

This is exactly the structure the Model Performance page reads: headline scores,
per-class table, confusion matrix, one-vs-rest ROC curves, and the model
comparison. The ROC curves are downsampled so the file stays small.

In [ ]:
report = classification_report(y_test, y_pred, target_names=labels,
                               output_dict=True, zero_division=0)
per_class = [
    {"class": name,
     "precision": round(vals["precision"], 4),
     "recall":    round(vals["recall"], 4),
     "f1":        round(vals["f1-score"], 4),
     "support":   int(vals["support"])}
    for name, vals in report.items() if name in labels
]

# One-vs-rest ROC, downsampled to keep metrics.json small.
y_bin = label_binarize(y_test, classes=range(len(labels)))
roc = {}
for i, name in enumerate(labels):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
    step = max(1, len(fpr) // 200)
    roc[name] = {"fpr": fpr[::step].tolist(),
                 "tpr": tpr[::step].tolist(),
                 "auc": float(auc(fpr, tpr))}

metrics = {
    "model_name": type(best_model).__name__,
    "trained_at": time.strftime("%Y-%m-%d"),
    "dataset": "CICIDS2017",
    "test_size": int(len(y_test)),
    "overall": {"accuracy":  float(acc),
                "precision": float(precision),
                "recall":    float(recall),
                "f1":        float(f1)},
    "per_class": per_class,
    "confusion_matrix": {"labels": labels,
                         "matrix": confusion_matrix(y_test, y_pred).tolist()},
    "roc": roc,
    "comparison": comparison,
}
print("metrics.json built for", metrics["model_name"])

## 12. Explainability with SHAP

A confusion matrix tells you *how often* the model is right. It does not tell you
*why* the model calls a flow an attack. SHAP does.

`TreeExplainer` computes, for every prediction, how much each of the 20 features
pushed the decision toward or away from each class. Averaged over many flows it
gives an honest global picture of what the model actually relies on; for a single
flow it explains that one decision.

This is what turns the project from "a classifier" into "an *explainable* IDS" —
the same reasoning a security analyst would want before trusting an alert. The
summary is exported as `shap_summary.json`, and the app can surface the top
contributing features next to each detection.

In [ ]:
# SHAP on the full test set is slow; a sample of ~500 flows is enough for a
# stable global picture and keeps this cell to well under a minute.
sample_n = min(500, len(X_test_scaled))
sample_idx = np.random.RandomState(42).choice(len(X_test_scaled), sample_n, replace=False)
X_sample = X_test_scaled[sample_idx]

print(f"Explaining {sample_n} flows with SHAP...")
explainer = shap.TreeExplainer(best_model)
shap_values = np.array(explainer.shap_values(X_sample))
print("SHAP values shape:", shap_values.shape)   # (samples, features, classes)

# Global importance: mean absolute SHAP value across samples and classes.
if shap_values.ndim == 3:
    global_importance = np.abs(shap_values).mean(axis=(0, 2))
else:
    global_importance = np.abs(shap_values).mean(axis=0)

order = np.argsort(global_importance)[::-1]
print("\nTop features by SHAP importance:")
for i in order[:10]:
    print(f"   {FEATURE_COLUMNS[i]:30} {global_importance[i]:.4f}")

In [ ]:
# Explain a single flow: which features pushed it toward its predicted class.
# Pick the first flow the model flagged as an attack, so the explanation is of
# something interesting rather than benign traffic.
sample_preds = best_model.predict(X_sample)
attack_positions = [i for i, p in enumerate(sample_preds)
                    if label_encoder.classes_[p] != "BENIGN"]

if attack_positions:
    pos = attack_positions[0]
    pred_class = sample_preds[pos]
    class_name = label_encoder.classes_[pred_class]
    contrib = shap_values[pos, :, pred_class] if shap_values.ndim == 3 else shap_values[pos]
    top = np.argsort(np.abs(contrib))[::-1][:6]

    print(f"Why this flow was classified as {class_name}:\n")
    for i in top:
        direction = "pushes toward" if contrib[i] > 0 else "pushes away from"
        print(f"   {FEATURE_COLUMNS[i]:30} {direction} {class_name:12} ({contrib[i]:+.3f})")
else:
    print("No attacks in the sample to explain (try a larger sample_n).")

In [ ]:
# A SHAP summary plot: the standard XAI visual. Save it as an image too, so it
# can go straight into a report or the repo.
import matplotlib.pyplot as plt

# Flatten to 2D (samples x features) by averaging absolute impact over classes,
# which is what a multiclass summary bar shows.
if shap_values.ndim == 3:
    mean_abs = np.abs(shap_values).mean(axis=2)
else:
    mean_abs = np.abs(shap_values)

shap.summary_plot(mean_abs, features=X_sample,
                  feature_names=FEATURE_COLUMNS, plot_type="bar", show=False)
plt.tight_layout()
shap_png = os.path.join(OUT_DIR, "shap_summary.png")
plt.savefig(shap_png, dpi=120, bbox_inches="tight")
plt.show()
print("Saved", shap_png)

In [ ]:
# Export the SHAP global importance so the app can show it beside detections.
shap_summary = {
    "method": "SHAP TreeExplainer",
    "sample_size": int(sample_n),
    "global_importance": [
        {"feature": FEATURE_COLUMNS[i],
         "importance": round(float(global_importance[i]), 5)}
        for i in order
    ],
}
with open(os.path.join(OUT_DIR, "shap_summary.json"), "w") as f:
    json.dump(shap_summary, f, indent=2)
print("shap_summary.json written to", OUT_DIR)

## 13. Export the files

These are the files the app loads. The cell also does a **self-check**: it
reloads them and reproduces a prediction, so you know the export is good before
you download anything.

In [ ]:
OUT = OUT_DIR   # on Drive
os.makedirs(OUT, exist_ok=True)

joblib.dump(best_model, f"{OUT}/model.joblib")
joblib.dump(scaler,     f"{OUT}/scaler.joblib")
joblib.dump(label_encoder, f"{OUT}/label_encoder.joblib")

with open(f"{OUT}/feature_columns.json", "w") as f:
    json.dump(FEATURE_COLUMNS, f, indent=2)
with open(f"{OUT}/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# shap_summary.json alongside the model files (png is already written here).
with open(f"{OUT}/shap_summary.json", "w") as f:
    json.dump(shap_summary, f, indent=2)

for fn in os.listdir(OUT):
    size = os.path.getsize(f"{OUT}/{fn}")
    print(f"  {fn:24} {size/1024:8.1f} KB")

model_mb = os.path.getsize(f"{OUT}/model.joblib") / 1024 / 1024
print(f"\nmodel.joblib is {model_mb:.1f} MB")
if model_mb > 90:
    print("  WARNING: close to GitHub's 100 MB limit. Lower n_estimators or "
          "max_depth and retrain, or use joblib compress.")

In [ ]:
# Self-check: reload the exports and reproduce a prediction, exactly as the app
# does — scaler.transform, model.predict, predict_proba, inverse_transform.
m  = joblib.load(f"{OUT}/model.joblib")
sc = joblib.load(f"{OUT}/scaler.joblib")
le = joblib.load(f"{OUT}/label_encoder.joblib")
cols = json.load(open(f"{OUT}/feature_columns.json"))

assert cols == FEATURE_COLUMNS, "feature order mismatch!"

sample = X_test.iloc[:5]
scaled = sc.transform(sample)
pred = le.inverse_transform(m.predict(scaled))
conf = m.predict_proba(scaled).max(axis=1)

print("Reload self-check:")
for p, c in zip(pred, conf):
    print(f"   {p:14} conf {c:.3f}")
print("\nExports are valid and reload cleanly.")

## 14. Download and plug in

Download the five files:

In [ ]:
from google.colab import files
for fn in ["model.joblib", "scaler.joblib", "label_encoder.joblib",
           "feature_columns.json", "metrics.json", "shap_summary.json"]:
    files.download(f"{OUT}/{fn}")

# The SHAP plot is a bonus for your report / slides, not needed by the app.
if os.path.exists(f"{OUT}/shap_summary.png"):
    files.download(f"{OUT}/shap_summary.png")

**Plug them into the app**

1. Put all five files in the app's `nids/models/` folder.
2. Start the app, go to **Settings / About → Model → Reload model**.

The sidebar changes from "Simulation mode" to the model name, every prediction
becomes real, and the Model Performance page fills in.

**To deploy the model** (only if the app is on GitHub / Streamlit Cloud):

```bash
git add -f nids/models/model.joblib nids/models/scaler.joblib \
           nids/models/label_encoder.joblib \
           nids/models/feature_columns.json nids/models/metrics.json
git commit -m "Add trained model"
git push
```

If `model.joblib` is over 100 MB, GitHub rejects it — retrain with a smaller
`n_estimators` or `max_depth`, which is why XGBoost with the settings above is
the safe default.